# 03 — SQL Analysis
## Smart Expense Intelligence Dashboard
**Goal:** Load clean data into MySQL, build schema, and write 8 advanced queries using CTEs, Window Functions, and LAG to extract business insights.

In [1]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# MySQL connection — update password to yours
DB_USER     = "root"
DB_PASSWORD = "root"
DB_HOST     = "localhost"
DB_NAME     = "expense_db"

# Create database first if not exists
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD
)
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
cursor.close()
conn.close()

# SQLAlchemy engine
engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

print("✅ Connected to MySQL successfully")
print(f"   Database: {DB_NAME}")

✅ Connected to MySQL successfully
   Database: expense_db


In [2]:
# Load clean CSV
df = pd.read_csv('../data/clean_transactions.csv', parse_dates=['date'])

# Load into MySQL — replaces table if exists
df.to_sql(
    name='transactions',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)

# Verify
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM transactions"))
    count = result.fetchone()[0]

print(f"✅ Data loaded into MySQL")
print(f"   Table: transactions")
print(f"   Rows : {count:,}")

✅ Data loaded into MySQL
   Table: transactions
   Rows : 50,000


In [3]:
def run_query(sql, title=""):
    with engine.connect() as conn:
        df_result = pd.read_sql(text(sql), conn)
    if title:
        print(f"\n{'='*55}")
        print(f"  {title}")
        print(f"{'='*55}")
    print(df_result.to_string(index=False))
    return df_result

print("✅ Helper function ready")

✅ Helper function ready


In [4]:
q1 = """
SELECT 
    category,
    COUNT(*)                                    AS total_transactions,
    ROUND(SUM(transaction_amount), 2)           AS total_spend,
    ROUND(AVG(transaction_amount), 2)           AS avg_transaction,
    ROUND(SUM(transaction_amount) * 100.0 / 
          (SELECT SUM(transaction_amount) 
           FROM transactions), 2)               AS spend_percentage
FROM transactions
GROUP BY category
ORDER BY total_spend DESC;
"""

df_q1 = run_query(q1, "Q1 — Total Spend by Category")


  Q1 — Total Spend by Category
   category  total_transactions  total_spend  avg_transaction  spend_percentage
     Travel                8377  12900231.94          1539.96             58.36
Electronics                8324   4394092.10           527.88             19.88
     Market                8382   2151134.42           256.64              9.73
   Clothing                8261   1319342.47           159.71              5.97
   Cosmetic                8243    876672.44           106.35              3.97
 Restaurant                8413    464488.60            55.21              2.10


In [5]:
q2 = """
WITH monthly_spend AS (
    SELECT 
        month,
        month_name,
        ROUND(SUM(transaction_amount), 2) AS total_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month, month_name
)
SELECT 
    month,
    month_name,
    total_spend,
    LAG(total_spend) OVER (ORDER BY month)   AS prev_month_spend,
    ROUND(total_spend - LAG(total_spend) 
          OVER (ORDER BY month), 2)          AS mom_change,
    ROUND((total_spend - LAG(total_spend) 
          OVER (ORDER BY month)) * 100.0 / 
          LAG(total_spend) 
          OVER (ORDER BY month), 2)          AS mom_pct_change
FROM monthly_spend
ORDER BY month;
"""

df_q2 = run_query(q2, "Q2 — Month over Month Spend Change (LAG)")


  Q2 — Month over Month Spend Change (LAG)
 month month_name  total_spend  prev_month_spend  mom_change  mom_pct_change
     1    January   2345228.21               NaN         NaN             NaN
     2   February   2148030.95        2345228.21  -197197.26           -8.41
     3      March   2396963.31        2148030.95   248932.36           11.59
     4      April   2289731.30        2396963.31  -107232.01           -4.47
     5        May   2460816.56        2289731.30   171085.26            7.47
     6       June   2254592.41        2460816.56  -206224.15           -8.38
     7       July   2475041.53        2254592.41   220449.12            9.78
     8     August   2391450.20        2475041.53   -83591.33           -3.38
     9  September   2274245.66        2391450.20  -117204.54           -4.90


In [6]:
q3 = """
WITH monthly AS (
    SELECT 
        month,
        month_name,
        ROUND(SUM(transaction_amount), 2) AS monthly_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month, month_name
)
SELECT
    month,
    month_name,
    monthly_spend,
    ROUND(SUM(monthly_spend) 
          OVER (ORDER BY month), 2)        AS running_total,
    ROUND(SUM(monthly_spend) 
          OVER (ORDER BY month) * 100.0 /
          SUM(monthly_spend) OVER (), 2)   AS cumulative_pct
FROM monthly
ORDER BY month;
"""

df_q3 = run_query(q3, "Q3 — Running Total Spend (Cumulative SUM)")


  Q3 — Running Total Spend (Cumulative SUM)
 month month_name  monthly_spend  running_total  cumulative_pct
     1    January     2345228.21     2345228.21           11.15
     2   February     2148030.95     4493259.16           21.36
     3      March     2396963.31     6890222.47           32.75
     4      April     2289731.30     9179953.77           43.64
     5        May     2460816.56    11640770.33           55.34
     6       June     2254592.41    13895362.74           66.05
     7       July     2475041.53    16370404.27           77.82
     8     August     2391450.20    18761854.47           89.19
     9  September     2274245.66    21036100.13          100.00


In [7]:
q4 = """
WITH monthly_cat AS (
    SELECT 
        month_name,
        month,
        category,
        ROUND(SUM(transaction_amount), 2) AS total_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month_name, month, category
)
SELECT
    month_name,
    category,
    total_spend,
    RANK() OVER (
        PARTITION BY month 
        ORDER BY total_spend DESC
    )                                      AS spend_rank
FROM monthly_cat
ORDER BY month, spend_rank;
"""

df_q4 = run_query(q4, "Q4 — Category Spend Rank per Month (RANK)")


  Q4 — Category Spend Rank per Month (RANK)
month_name    category  total_spend  spend_rank
   January      Travel   1329824.90           1
   January Electronics    477148.03           2
   January      Market    244372.68           3
   January    Clothing    146401.58           4
   January    Cosmetic     97668.85           5
   January  Restaurant     49812.17           6
  February      Travel   1241760.61           1
  February Electronics    440793.89           2
  February      Market    208058.58           3
  February    Clothing    122817.37           4
  February    Cosmetic     91017.27           5
  February  Restaurant     43583.23           6
     March      Travel   1417468.00           1
     March Electronics    477311.72           2
     March      Market    217344.73           3
     March    Clothing    137858.96           4
     March    Cosmetic     95094.24           5
     March  Restaurant     51885.66           6
     April      Travel   1351148.33        

In [8]:
q5 = """
WITH customer_spend AS (
    SELECT
        customer_id,
        name,
        surname,
        gender,
        age,
        COUNT(*)                            AS total_transactions,
        ROUND(SUM(transaction_amount), 2)   AS total_spend,
        ROUND(AVG(transaction_amount), 2)   AS avg_transaction
    FROM transactions
    GROUP BY customer_id, name, surname, gender, age
)
SELECT
    customer_id,
    name,
    surname,
    gender,
    age,
    total_transactions,
    total_spend,
    avg_transaction,
    DENSE_RANK() OVER (
        ORDER BY total_spend DESC
    )                                       AS spend_rank
FROM customer_spend
ORDER BY spend_rank
LIMIT 10;
"""

df_q5 = run_query(q5, "Q5 — Top 10 Customers by Total Spend (DENSE_RANK)")


  Q5 — Top 10 Customers by Total Spend (DENSE_RANK)
 customer_id        name  surname gender  age  total_transactions  total_spend  avg_transaction  spend_rank
      281313   Elizabeth     Lutz      M 61.0                   1      2999.88          2999.88           1
       55197 Christopher   Foster      M 60.0                   1      2999.68          2999.68           2
      119569       Billy Williams      F 34.0                   1      2999.22          2999.22           3
       56330     Madison    Davis      M 63.0                   1      2998.51          2998.51           4
        8255      Joseph  Wallace      F 46.0                   1      2998.48          2998.48           5
       50769       Kelly     Hall      M 22.0                   1      2997.81          2997.81           6
      102093        Noah   Graham      M 51.0                   1      2997.11          2997.11           7
      174847     Shannon   Torres      F 44.0                   1      2996.86     

In [9]:
q6 = """
WITH base AS (
    SELECT
        gender,
        category,
        ROUND(SUM(transaction_amount), 2) AS total_spend
    FROM transactions
    WHERE gender != 'Unknown'
    GROUP BY gender, category
)
SELECT
    category,
    MAX(CASE WHEN gender = 'F' THEN total_spend END) AS female_spend,
    MAX(CASE WHEN gender = 'M' THEN total_spend END) AS male_spend,
    ROUND(MAX(CASE WHEN gender = 'F' THEN total_spend END) -
          MAX(CASE WHEN gender = 'M' THEN total_spend END), 2) AS difference,
    CASE 
        WHEN MAX(CASE WHEN gender = 'F' THEN total_spend END) >
             MAX(CASE WHEN gender = 'M' THEN total_spend END) 
        THEN 'Female Higher'
        ELSE 'Male Higher'
    END                                              AS winner
FROM base
GROUP BY category
ORDER BY female_spend DESC;
"""

df_q6 = run_query(q6, "Q6 — Spend by Gender and Category (Pivot CTE)")


  Q6 — Spend by Gender and Category (Pivot CTE)
   category  female_spend  male_spend  difference        winner
     Travel    5975346.51  5656689.12   318657.39 Female Higher
Electronics    1977312.00  1994008.68   -16696.68   Male Higher
     Market     959849.62   960420.88     -571.26   Male Higher
   Clothing     587938.88   594994.60    -7055.72   Male Higher
   Cosmetic     407487.58   383398.72    24088.86 Female Higher
 Restaurant     211185.65   205370.81     5814.84 Female Higher


In [10]:
q7 = """
WITH spend_type AS (
    SELECT
        CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        category,
        COUNT(*)                            AS transactions,
        ROUND(SUM(transaction_amount), 2)   AS total_spend,
        ROUND(AVG(transaction_amount), 2)   AS avg_spend
    FROM transactions
    GROUP BY day_type, category
)
SELECT
    day_type,
    category,
    transactions,
    total_spend,
    avg_spend,
    ROUND(total_spend * 100.0 / 
          SUM(total_spend) OVER (
              PARTITION BY day_type
          ), 2)                             AS pct_within_day_type
FROM spend_type
ORDER BY day_type, total_spend DESC;
"""

df_q7 = run_query(q7, "Q7 — Weekend vs Weekday Spend by Category")


  Q7 — Weekend vs Weekday Spend by Category
day_type    category  transactions  total_spend  avg_spend  pct_within_day_type
 Weekday      Travel          5990   9221082.62    1539.41                58.30
 Weekday Electronics          5956   3148833.60     528.68                19.91
 Weekday      Market          6021   1542372.33     256.17                 9.75
 Weekday    Clothing          5896    947757.16     160.75                 5.99
 Weekday    Cosmetic          5922    629032.62     106.22                 3.98
 Weekday  Restaurant          5946    328330.71      55.22                 2.08
 Weekend      Travel          2387   3679149.32    1541.33                58.51
 Weekend Electronics          2368   1245258.50     525.87                19.80
 Weekend      Market          2361    608762.09     257.84                 9.68
 Weekend    Clothing          2365    371585.31     157.12                 5.91
 Weekend    Cosmetic          2321    247639.82     106.70                 

In [11]:
q8 = """
WITH monthly_avg AS (
    SELECT
        month,
        month_name,
        ROUND(AVG(transaction_amount), 2)    AS avg_spend,
        ROUND(STDDEV(transaction_amount), 2) AS std_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month, month_name
),
anomalies AS (
    SELECT
        t.month,
        t.month_name,
        COUNT(*) AS anomaly_count,
        ROUND(SUM(t.transaction_amount), 2)  AS anomaly_total,
        ROUND(AVG(t.transaction_amount), 2)  AS anomaly_avg
    FROM transactions t
    JOIN monthly_avg m ON t.month = m.month
    WHERE t.transaction_amount > m.avg_spend + (2 * m.std_spend)
    AND t.month < 10
    GROUP BY t.month, t.month_name
)
SELECT
    month,
    month_name,
    anomaly_count,
    anomaly_total,
    anomaly_avg,
    RANK() OVER (ORDER BY anomaly_count DESC) AS anomaly_rank
FROM anomalies
ORDER BY month;
"""

df_q8 = run_query(q8, "Q8 — Monthly Anomaly Detection (2+ Std Dev)")


  Q8 — Monthly Anomaly Detection (2+ Std Dev)
 month month_name  anomaly_count  anomaly_total  anomaly_avg  anomaly_rank
     1    January            386      891846.96      2310.48             5
     2   February            362      842039.20      2326.08             9
     3      March            401      926762.31      2311.13             4
     4      April            385      910224.38      2364.22             6
     5        May            414      989040.38      2388.99             2
     6       June            374      863577.15      2309.03             8
     7       July            422     1000643.94      2371.19             1
     8     August            402      933141.46      2321.25             3
     9  September            385      894565.40      2323.55             6


In [12]:
queries = """
-- =====================================================
-- Smart Expense Intelligence Dashboard
-- SQL Analysis Queries
-- =====================================================

-- Q1: Total Spend by Category
SELECT 
    category,
    COUNT(*)                                    AS total_transactions,
    ROUND(SUM(transaction_amount), 2)           AS total_spend,
    ROUND(AVG(transaction_amount), 2)           AS avg_transaction,
    ROUND(SUM(transaction_amount) * 100.0 / 
          (SELECT SUM(transaction_amount) 
           FROM transactions), 2)               AS spend_percentage
FROM transactions
GROUP BY category
ORDER BY total_spend DESC;

-- Q2: Month over Month Spend Change (LAG)
WITH monthly_spend AS (
    SELECT 
        month, month_name,
        ROUND(SUM(transaction_amount), 2) AS total_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month, month_name
)
SELECT 
    month, month_name, total_spend,
    LAG(total_spend) OVER (ORDER BY month)   AS prev_month_spend,
    ROUND(total_spend - LAG(total_spend) 
          OVER (ORDER BY month), 2)          AS mom_change,
    ROUND((total_spend - LAG(total_spend) 
          OVER (ORDER BY month)) * 100.0 / 
          LAG(total_spend) 
          OVER (ORDER BY month), 2)          AS mom_pct_change
FROM monthly_spend
ORDER BY month;

-- Q3: Running Total Spend (Cumulative SUM)
WITH monthly AS (
    SELECT month, month_name,
        ROUND(SUM(transaction_amount), 2) AS monthly_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month, month_name
)
SELECT month, month_name, monthly_spend,
    ROUND(SUM(monthly_spend) OVER (ORDER BY month), 2) AS running_total,
    ROUND(SUM(monthly_spend) OVER (ORDER BY month) * 100.0 /
          SUM(monthly_spend) OVER (), 2)               AS cumulative_pct
FROM monthly
ORDER BY month;

-- Q4: Category Spend Rank per Month (RANK)
WITH monthly_cat AS (
    SELECT month_name, month, category,
        ROUND(SUM(transaction_amount), 2) AS total_spend
    FROM transactions
    WHERE month < 10
    GROUP BY month_name, month, category
)
SELECT month_name, category, total_spend,
    RANK() OVER (
        PARTITION BY month ORDER BY total_spend DESC
    ) AS spend_rank
FROM monthly_cat
ORDER BY month, spend_rank;

-- Q5: Top 10 Customers by Total Spend (DENSE_RANK)
WITH customer_spend AS (
    SELECT customer_id, name, surname, gender, age,
        COUNT(*)                          AS total_transactions,
        ROUND(SUM(transaction_amount), 2) AS total_spend,
        ROUND(AVG(transaction_amount), 2) AS avg_transaction
    FROM transactions
    GROUP BY customer_id, name, surname, gender, age
)
SELECT *, DENSE_RANK() OVER (ORDER BY total_spend DESC) AS spend_rank
FROM customer_spend
ORDER BY spend_rank LIMIT 10;

-- Q6: Spend by Gender and Category (Pivot CTE)
WITH base AS (
    SELECT gender, category,
        ROUND(SUM(transaction_amount), 2) AS total_spend
    FROM transactions
    WHERE gender != 'Unknown'
    GROUP BY gender, category
)
SELECT category,
    MAX(CASE WHEN gender = 'F' THEN total_spend END) AS female_spend,
    MAX(CASE WHEN gender = 'M' THEN total_spend END) AS male_spend,
    ROUND(MAX(CASE WHEN gender = 'F' THEN total_spend END) -
          MAX(CASE WHEN gender = 'M' THEN total_spend END), 2) AS difference,
    CASE WHEN MAX(CASE WHEN gender = 'F' THEN total_spend END) >
              MAX(CASE WHEN gender = 'M' THEN total_spend END)
         THEN 'Female Higher' ELSE 'Male Higher' END AS winner
FROM base
GROUP BY category
ORDER BY female_spend DESC;

-- Q7: Weekend vs Weekday Spend by Category
WITH spend_type AS (
    SELECT CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        category,
        COUNT(*)                          AS transactions,
        ROUND(SUM(transaction_amount), 2) AS total_spend,
        ROUND(AVG(transaction_amount), 2) AS avg_spend
    FROM transactions
    GROUP BY day_type, category
)
SELECT day_type, category, transactions, total_spend, avg_spend,
    ROUND(total_spend * 100.0 / 
          SUM(total_spend) OVER (PARTITION BY day_type), 2) AS pct_within_day_type
FROM spend_type
ORDER BY day_type, total_spend DESC;

-- Q8: Monthly Anomaly Detection (2+ Std Dev)
WITH monthly_avg AS (
    SELECT month, month_name,
        ROUND(AVG(transaction_amount), 2)    AS avg_spend,
        ROUND(STDDEV(transaction_amount), 2) AS std_spend
    FROM transactions WHERE month < 10
    GROUP BY month, month_name
),
anomalies AS (
    SELECT t.month, t.month_name,
        COUNT(*) AS anomaly_count,
        ROUND(SUM(t.transaction_amount), 2) AS anomaly_total,
        ROUND(AVG(t.transaction_amount), 2) AS anomaly_avg
    FROM transactions t
    JOIN monthly_avg m ON t.month = m.month
    WHERE t.transaction_amount > m.avg_spend + (2 * m.std_spend)
    AND t.month < 10
    GROUP BY t.month, t.month_name
)
SELECT month, month_name, anomaly_count, anomaly_total, anomaly_avg,
    RANK() OVER (ORDER BY anomaly_count DESC) AS anomaly_rank
FROM anomalies ORDER BY month;
"""

# Save to file
import os
os.makedirs('../sql', exist_ok=True)
with open('../sql/queries.sql', 'w') as f:
    f.write(queries)

print("✅ All 8 queries saved → sql/queries.sql")

✅ All 8 queries saved → sql/queries.sql
